# Getting Started with Kimi K3 on Amazon Bedrock

This notebook walks through how to get started using [Kimi K3](https://www.kimi.com/blog/kimi-k3), a frontier-intelligence open weight model from [Moonshot AI](https://www.moonshot.ai/), on [Amazon Bedrock](https://aws.amazon.com/bedrock/).

## Contents

- Prerequisites
- Setup
- Supported inference APIs
    - Cross-region inference
    - OpenAI-compatible API examples
    - Native Amazon Bedrock API examples
- AI framework libraries
    - LangChain examples
    - Strands Agents examples
- Feature call-outs
    - Prompt caching
- Conclusion

## Prerequisites<a id="0"></a>

To run this notebook, you'll need:
- Access to an [AWS Account](https://aws.amazon.com/resources/create-account/), with sufficient permissions to invoke Kimi models on Amazon Bedrock
    - Whether you log in directly as an [IAM User](https://docs.aws.amazon.com/IAM/latest/UserGuide/id_users.html) or assume a [Role](https://docs.aws.amazon.com/IAM/latest/UserGuide/id_roles.html), either way you'll need to make sure your identity has attached [policies](https://docs.aws.amazon.com/IAM/latest/UserGuide/access_policies.html) to grant Bedrock access.
    - To get started quickly, you can attach the AWS-Managed `AmazonBedrockLimitedAccess` policy which grants broad access across models. For production environments, we recommend reviewing and scoping down required access following the [principle of least-privilege](https://docs.aws.amazon.com/IAM/latest/UserGuide/getting-started-reduce-permissions.html).
    - See the [model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards-moonshot-ai.html) for up-to-date information on which [AWS Regions](https://aws.amazon.com/about-aws/global-infrastructure/regions_az/) where Kimi K3 is available.
- A development environment to run this notebook, with [Python](https://www.python.org/) (v3.12+) and the [AWS CLI configured](https://docs.aws.amazon.com/cli/latest/userguide/cli-chap-configure.html) with your AWS IAM User/Role/Profile.
    - You could use a local IDE like [Kiro](https://kiro.dev/) or [VS Code](https://code.visualstudio.com/) on your own computer...
    - ...Or a Cloud environment like [Amazon SageMaker AI Studio](https://aws.amazon.com/sagemaker/ai/studio/) - which will already be configured with an IAM Role and Region when you create it.

## Setup

You'll need to install the required libraries and **select your Python kernel environment** to run this notebook.

**For local IDEs**, we recommend using [uv](https://docs.astral.sh/uv/) to manage your Python environments. After installing it, you can open a terminal in this repository's root folder and run:
1. `uv venv --allow-existing` to create the virtual environment (usually in a new folder `.env` in the project itself), then
2. `uv sync` to install the libraries.

When the virtual environment is set up, select it as the kernel for this notebook (In VS Code or Kiro, choose `Python Environments > .venv` when prompted).

You can also use plain `venv` if you prefer, and `pip install -e .` to install the dependencies (which are specified in this repository's top-level [pyproject.toml](../pyproject.toml)).

**For SageMaker JupyterLab Notebooks**, you can use the base `conda_python3` kernel and install the dependencies through pip, by un-commenting and running the code cell below:

In [ ]:
# (For SageMaker JuptyerLab)
# %pip install -e ..

## Supported inference APIs

Amazon Bedrock supports [multiple inference APIs](https://docs.aws.amazon.com/bedrock/latest/userguide/apis.html), giving customers choice between AWS' own native APIs as well as endpoints that are compatible with other providers.

For up-to-date information on which API endpoints are supported for this model, see the [Kimi K3 model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards-moonshot-ai.html) in the Amazon Bedrock User Guide. At the time of writing, both the **OpenAI-compatible** and **native Amazon Bedrock** endpoints are supported.

### Cross-region inference

We recommend to use Kimi K3 through global [cross-region inference](https://docs.aws.amazon.com/bedrock/latest/userguide/cross-region-inference.html) where possible, for lower cost and faster end-to-end processing times (especially during local busy periods).

This is configured by attaching a cross-region prefix to the model ID. In the below example we'll use `global.`, but other prefixes like `us.`, `in.` may also be supported. Check the [Kimi K3 model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-moonshot-ai-kimi-k3.html) in the Amazon Bedrock User Guide for the latest on what inference profiles are available.

In [ ]:
model_id = "global.moonshotai.kimi-k3"

### OpenAI-compatible API examples

Amazon Bedrock's OpenAI-compatible [Chat Completions](https://docs.aws.amazon.com/bedrock/latest/userguide/inference-chat-completions-mantle.html) and [Responses](https://docs.aws.amazon.com/bedrock/latest/userguide/bedrock-mantle.html) APIs provide drop-in replacement for applications that are already using OpenAI or other compatible providers.

As shown in the below example with the OpenAI Python SDK, you'll need to configure your `base_url` to point to Amazon Bedrock's regional endpoint, and your `api_key` with access credentials.

It *is possible* to [generate long-lived API keys for Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/api-keys-generate.html). However, as a security best-practice we recommend to use short-lived credentials instead where possible. For Python applications, the [aws-bedrock-token-generator](https://pypi.org/project/aws-bedrock-token-generator/) library can render a bearer token directly from your ambient AWS CLI / IAM credentials - without you needing to create and manage the lifecycle of an API key. For example:

In [ ]:
import os

from aws_bedrock_token_generator import provide_token
from openai import OpenAI

region = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")

oai_client = OpenAI(
    api_key=provide_token(region=region),
    base_url=f"https://bedrock-runtime.{region}.amazonaws.com/openai/v1",
)

With the OpenAI client set up, you can invoke the model through either the Chat Completions API or the Responses API as needed. Features like streaming, tool calling, and reasoning effort configuration are available as usual:

In [ ]:
cpl = oai_client.chat.completions.create(
    model=model_id,
    messages=[
        {
            "role": "user",
            "content": "Briefly, explain the CAP theorem?"
        }
    ],
)

print(cpl.choices[0].message.content)

In [ ]:
resp = oai_client.responses.create(
    model=model_id,
    input="What's 2 + 2?",
)

print(resp.output_text)

### Native Amazon Bedrock API examples

> ⚠️ **Warning:** Although the Converse API is supported for Kimi K3 on Amazon Bedrock, it has known issues and is **not recommended**. In particular, multi-turn requests that include *historical reasoning* in previous model turns will fail. See the [documented limitations in the model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-moonshot-ai-kimi-k3.html#model-card-moonshot-ai-kimi-k3-considerations) for more details.

The [Converse API](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_Converse.html) and its streaming equivalent [ConverseStream](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_ConverseStream.html) provide a common interface between open weight models and Anthropic Claude models (for which the OpenAI-compatible endpoints are not currently available).

Applications that are already using models on Amazon Bedrock through the Converse API, can directly switch to Kimi K3 by updating the target `modelId`, subject to the [limitations](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-moonshot-ai-kimi-k3.html#model-card-moonshot-ai-kimi-k3-considerations) of the API with this model.

Like other native AWS APIs, these can be accessed through the AWS SDKs for your programming language of choice. In this example we'll use boto3, the AWS SDK for Python:

In [ ]:
import json

import boto3

region = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")

br_client = boto3.client("bedrock-runtime", region_name=region)

In [ ]:
converse_resp = br_client.converse(
    modelId=model_id,
    messages=[
        {
            "content": [
                {"text": "Who created the 'Kimi' series of AI models?"},
            ],
            "role": "user",
        },
    ],
)

for blk in converse_resp["output"]["message"]["content"]:
    if "text" in blk:
        print(blk["text"])

## AI framework libraries

Many common AI application frameworks support either Amazon Bedrock native APIs, or OpenAI-compatible APIs - so can be used with Kimi K3 on Amazon Bedrock. We'll show some examples in the following sections.

### LangChain examples

LangChain and LangGraph support using either the native Amazon Bedrock Converse API (via [`ChatBedrockConverse`](https://reference.langchain.com/python/langchain-aws/chat_models/bedrock_converse/ChatBedrockConverse) from the [langchain-aws](https://github.com/langchain-ai/langchain-aws/) package), or the OpenAI-compatible APIs (by setting the `base_url` parameter to point to Amazon Bedrock).

As discussed above, the OpenAI-compatible Chat Completions API is recommended over the Converse API for Kimi K3 - so the preferred integration would look like the example below:

In [ ]:
from langchain_openai import ChatOpenAI

region = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")

llm = ChatOpenAI(
    model=model_id,
    api_key=provide_token(region),
    base_url=f"https://bedrock-runtime.{region}.amazonaws.com/openai/v1",
)

In [ ]:
llm_resp = llm.invoke("Briefly, what is cross-Region inference on Amazon Bedrock?")

print(llm_resp.text)

### Strands Agents examples

[Strands Agents SDK](https://strandsagents.com/) is an open source agent building framework that's easy to get started, thanks to its "model-first" approach, but still offers deep features and customization.

Like LangChain, it supports both native Amazon Bedrock APIs (through [`BedrockModel`](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/)) and OpenAI-compatible APIs (through e.g. [`OpenAIModel`](https://strandsagents.com/docs/api/python/strands.models.openai/) or [`OpenAIResponsesModel`](https://strandsagents.com/docs/user-guide/concepts/model-providers/openai-responses/)) - both of which can be used with Kimi K3 on Amazon Bedrock.

For example, using the preferred OpenAI-compatible API:

In [ ]:
from openai import AsyncOpenAI
from strands import Agent
from strands.models import OpenAIModel
from strands_tools import calculator

model = OpenAIModel(
    model_id=model_id,
    client=AsyncOpenAI(
        api_key=provide_token(region=region),
        base_url=f"https://bedrock-runtime.{region}.amazonaws.com/openai/v1",
    ),
)
agent = Agent(
    model=model,
    system_prompt="For reliable results, always use the calculator tool when asked a math question",
    tools=[calculator],
)

agent_resp = agent("What's the square root of 15,812,811,001?")
print(agent_resp.message)

## Feature call-outs

Generally, [Kimi K3 features](https://platform.kimi.ai/docs/guide/kimi-k3-quickstart) (like vision input, reasoning effort configuration, function and tool calling, and structured output) should work on Amazon Bedrock as described in Kimi's own documentation.

However, the following Kimi features are **not supported** at this time:

1. [Partial mode](https://platform.kimi.ai/docs/guide/use-partial-mode-feature-of-kimi-api) - providing the start of a response for the model to continue
2. [Dynamic tool loading](https://platform.kimi.ai/docs/guide/use-dynamic-tool-loading) within system messages during a conversation - Use the top-level `tools` field instead, to list available tools for each invocation.

It's also worth calling out an Amazon Bedrock-specific capability: **Explicit prompt caching**.

### Prompt caching

Like other open weight models hosted on Amazon Bedrock's Mantle platform, Kimi K3 supports **implicit prompt caching** to help accelerate responses with no user action required.

The cache status of each request is returned in the metadata of the response, so you can see it in action by invoking the model with the same prompt a few times and checking the metadata.

Prompt caching generally makes sense for use-cases with a long fixed prompt prefix, so for this example we'll try continuing an extract of a story, using the starter prompt in [util/story_prompt.py](./util/story_prompt.py):

In [ ]:
import os

from aws_bedrock_token_generator import provide_token
from openai import OpenAI

from util.story_prompt import PROMPT_BASE

region = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")

oai_client = OpenAI(
    api_key=provide_token(region=region),
    base_url=f"https://bedrock-runtime.{region}.amazonaws.com/openai/v1",
)

Note that **hitting cache is not guaranteed**, even when sending multiple matching prompts - because Amazon Bedrock manages trade-offs between routing your requests to running copies of the model that A/ have your matching prompt in cache, or B/ have a less busy queue of requests: Optimizing for fastest overall reponse time.

In [ ]:
max_attempts = 20
for i in range(1, max_attempts + 1):
    print(f"Request {i}... ", end="")
    resp = oai_client.responses.create(
        model="global.moonshotai.kimi-k3",
        input=PROMPT_BASE,
    )
    if resp.usage.input_tokens_details.cached_tokens:
        print("✅ Hit cache!")
        print(resp.usage)
        break
    else:
        print("❌ Missed cache")
    if i == max_attempts:
        # Should be unlikely - but possible:
        print("All attempts missed cache - please try again!")

Alongside this automatic caching, Amazon Bedrock gives you more precise control with [explicit prompt caching](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html#prompt-caching-openai) options.

You can control caching through 3 parameters on inference requests:
- `prompt_cache_breakpoint`: placed on a content block in your input messages, marks the exact end of a re-usable prompt prefix. Everything up to and including the marked block is cached, and everything after it can change freely while the cached prefix stays valid. Each cached prefix must contain at least 1,024 tokens, and you can set up to 4 breakpoints per request on `input_text`, `input_image`, and `input_file` blocks.
- `prompt_cache_key`: a stable ID that routes requests to the same cache. Set it consistently across all requests in a session or application so repeat requests find the cached prefix.
- `prompt_cache_options`: controls the caching mode (implicit or explicit) and the Time-To-Live (TTL) period after which the cache will expire.

This is particularly useful if you have, for example:
- A stable prefix plus changing suffix (such as chat assistants, RAG, or agentic tool loops), or
- Multiple sections of your prompt that change at different frequencies (which can be assigned separate cache break points)

In this simple example, we'll just continue our story with different style modifiers:

In [ ]:
import random

max_attempts = 20
random_words = ("marmalade", "Paddington", "hiccup", "tuppence")
for i in range(1, max_attempts + 1):
    word = random.choice(random_words)
    print(f"Request {i} (word '{word}')... ", end="")
    resp = oai_client.responses.create(
        model="global.moonshotai.kimi-k3",
        prompt_cache_key="story-completion-example",
        extra_body={"prompt_cache_options": {"mode": "explicit"}},
        input=[
            {
                "type": "message",
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": PROMPT_BASE,
                        "prompt_cache_breakpoint": {"mode": "explicit"},
                    },
                    {
                        "type": "input_text",
                        # This changing (but repeated) suffix can also be cached:
                        "text": f"\n\n(BUT: Be sure to mention the word '{word}'.) Continuation starts now:\n",
                        "prompt_cache_breakpoint": {"mode": "explicit"},
                    },
                ],
            },
        ],
    )
    if resp.usage.input_tokens_details.cached_tokens:
        print("✅ Hit cache!")
        print(resp.usage)
    else:
        print("❌ Missed cache")

## Conclusion

In this notebook we showed some basic Python code samples for getting started with Kimi K3 on Amazon Bedrock, with simple prompts to demonstrate the core functionalities.

K3 is a highly-intelligent model that has shown frontier-level performance on a range of benchmarks in code generation, long-context reasoning, and analysis. Although its weights have been published, at 2.8 trillion parameters the model can be challenging to self-host. We're looking forward to hear about the advanced applications customers like you build, with as-a-service access to use K3 on Amazon Bedrock!